# 01 — Your First CGE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miraflor/CGE-core/blob/main/notebooks/01_your_first_cge.ipynb)

**Model:** Hosoe, Gasawa & Hashimoto's pedagogical `splcge` model.

```text
Capital ─┐              ┌─ Bread ─┐
         ├─ production ─┤         │
Labor ───┘              └─ Milk ──┼─→ Household consumption
                                  │
Household owns Capital + Labor ───┘
```

There is **no government, no trade, no tax system, and no intermediate-input network**.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# In Colab, set the environment variable CGE_CORE_REF to test a branch or tag.
# The public notebooks default to main. Outside Colab, a current CGE-Core git
# checkout is used directly, so branch development never silently resets to main.
CGE_CORE_REF = os.environ.get("CGE_CORE_REF", "main")
IN_COLAB = Path("/content").exists()

if not IN_COLAB and (Path.cwd() / ".git").is_dir() and (Path.cwd() / "cge_core").is_dir():
    REPO_DIR = Path.cwd()
    source_label = "current checkout"
else:
    WORKSPACE = Path("/content") if IN_COLAB else Path.home() / ".cache"
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    REPO_DIR = WORKSPACE / "CGE-core-colab"
    REPO_URL = "https://github.com/miraflor/CGE-core.git"

    if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a git checkout.")
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--no-checkout", REPO_URL, str(REPO_DIR)],
            check=True,
        )

    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", CGE_CORE_REF, "--depth", "1"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"],
        check=True,
    )
    source_label = CGE_CORE_REF

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import cge_core

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
print("✓ CGE-Core", cge_core.__version__)
print("✓ Source:", source_label, f"({commit})")
print("✓ Repository:", REPO_DIR)


import shutil

if not shutil.which("ipopt"):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "amplpy"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "amplpy.modules", "install", "coin"],
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    module_path = subprocess.check_output(
        [sys.executable, "-m", "amplpy.modules", "path"],
        text=True,
    ).strip()
    os.environ["PATH"] = module_path + os.pathsep + os.environ.get("PATH", "")

assert shutil.which("ipopt"), "IPOPT was not found."
SOLVER = "ipopt"
print("✓ Solver:", SOLVER)


## 2. Load the benchmark economy

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from cge_core import CGE, example_data
from cge_core.models import SplCGE

data_dir = example_data("splcge")
sam = pd.read_csv(data_dir / "param-sam-.csv", index_col=0)
GOODS = ["BRD", "MLK"]
FACTORS = ["CAP", "LAB"]
display(sam)

## 3. Solve the benchmark equilibrium

In [ ]:
model = CGE(model=SplCGE(), data=data_dir)
benchmark = model.solve_benchmark(
    numeraire=("pf", "LAB"),
    redundant=("eqpf", "LAB"),
    solver=SOLVER,
)

benchmark_table = pd.DataFrame({
    "good": GOODS,
    "output_Z": [benchmark.value("Z", good) for good in GOODS],
    "consumption_X": [benchmark.value("X", good) for good in GOODS],
    "goods_price_px": [benchmark.value("px", good) for good in GOODS],
})
display(benchmark_table)

print("Factor prices:")
for factor in FACTORS:
    print(f"  {factor}: {benchmark.value('pf', factor):.4f}")
print("Benchmark welfare objective:", benchmark.objective)

## 4. Your first shock 👇

Give the household more of one factor. The default is **10% more capital**.

In [ ]:
# 👇 EDIT THESE
FACTOR = "CAP"          # "CAP" or "LAB"
FACTOR_CHANGE_PCT = 10

## 5. Solve the counterfactual economy

In [ ]:
if FACTOR not in FACTORS:
    raise ValueError(f"FACTOR must be one of {FACTORS}")

base_endowment = benchmark.value("FF", FACTOR)
new_endowment = base_endowment * (1 + FACTOR_CHANGE_PCT / 100)

scenario = benchmark.scenario(f"{FACTOR} endowment {FACTOR_CHANGE_PCT:+g}%")
scenario.set("FF", FACTOR, new_endowment)
result = scenario.solve(solver=SOLVER)
results = result.compare(benchmark)

print(f"{FACTOR} endowment: {base_endowment:.4f} → {new_endowment:.4f}")
print(f"Welfare: {benchmark.objective:.4f} → {result.objective:.4f}")

## 6. See the equilibrium response

In [ ]:
headline = results[results["component"].isin(["Z", "X", "pf", "px"])].copy()
display(
    headline[["component", "index_1", "reference_value", "value", "difference", "pct_change"]]
    .style.format({
        "reference_value": "{:.4f}",
        "value": "{:.4f}",
        "difference": "{:+.4f}",
        "pct_change": "{:+.2f}%",
    })
)

In [ ]:
for component, title in [
    ("Z", "Output"),
    ("X", "Household consumption"),
    ("pf", "Factor prices"),
]:
    part = results[results["component"] == component]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(part["index_1"].astype(str), part["pct_change"].astype(float))
    ax.axhline(0, linewidth=0.8)
    ax.set_title(title)
    ax.set_ylabel("% change from benchmark")
    plt.show()

## What you learned

You changed one exogenous resource endowment. The model then found a new equilibrium in which firms optimize, the household optimizes, and goods and factor markets clear simultaneously.

## Next

Notebook 02 adds government, taxes, imports, exports, saving, and investment.

[Open Notebook 02 in Colab](https://colab.research.google.com/github/miraflor/CGE-core/blob/main/notebooks/02_open_economy_cge.ipynb)